# Klassifikation mit Gradientenabstieg

Wir haben die Klassifikation (*kommt in den nächsten Tagen*) noch nicht behandelt, aber wir können ```Gradient Descent``` auch für die Klassifikation verwenden. In diesem Notebook können Sie sich ansehen, wie man das macht, und können später darauf zurückkommen, um es mit den anderen Methoden zu vergleichen. Wir werden dieses Beispiel auch verwenden, um SGD und Mini-Batch GD zu visualisieren.

## Visualisierung von Stochastic Gradient Descent (SGD) und Mini Batch Gradient Descent
Jetzt schauen wir uns Beispiele für die Visualisierung des Stochastic Gradient Descent (SGD) und des Mini Batch Gradient Descent mit dem scikit-learn ```sklearn.linear_model.SGDClassifier()``` an. Zuerst laden wir den digits-Datensatz, der aus 8x8-Bildern einer Ziffer (0 und 1) besteht.

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import sklearn.datasets as dt
import numpy as np

In [ ]:
# Load the digits dataset with two classes
digits, target = dt.load_digits(n_class=2, return_X_y=True)
fig, ax = plt.subplots(nrows=1, ncols=10, figsize=(12,4), subplot_kw=dict(xticks=[], yticks=[]))

# Plot some images of digits
for i in np.arange(10):
    ax[i].imshow(digits[i,:].reshape(8,8), cmap=plt.cm.grey, interpolation='nearest')   
plt.show()

In [ ]:
# Each row of 'digits ' is an image of 8x8 pixels (64 features) of a handwritten digit
print(digits.shape)  # (n_samples, n_features)
print(target.shape)  # (n_samples,)
# Target contains the labels (0 or 1)
target


Jetzt teilen wir den Datensatz in Trainings- und Testset auf.

In [ ]:
# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(
                        digits, target, test_size=0.2, random_state=10)

## Visualisierung von Stochastic Gradient Descent (SGD)

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
import sys
from io import StringIO
import matplotlib.pyplot as plt

In [ ]:
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()

# You can look up the documentation and play with the hyperparameters
clf = SGDClassifier(loss='log_loss', penalty="l1", alpha= 0.1, verbose=1, n_jobs=-1, random_state=10)
clf.fit(X_train_sc, y_train)

# Plotting the loss function
sys.stdout = old_stdout
loss_history = mystdout.getvalue()
loss_list = []
for line in loss_history.split('\n'):
    if(len(line.split("loss: ")) == 1):
        continue
    loss_list.append(float(line.split("loss: ")[-1].split(",")[0]))
plt.figure()
plt.plot(np.arange(len(loss_list)), loss_list)
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()


In [ ]:
accuracy = clf.score(X_test_sc, y_test)
print(f"Accuracy: {round(accuracy, 2)}")

Sie können sehen, dass das Modell mit dem SGD ziemlich schnell besser wird. Und eine Genauigkeit von 97%.

## Visualisierung von Mini Batch Gradient Descent

In [ ]:
old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()

# You can look up the documentation and play with the hyperparameters
clf = SGDClassifier(loss='log_loss', penalty="l1", alpha= 0.1, verbose=1, n_jobs=-1, random_state=10)



# creating the batches N = 6 (the shape of X_train is 288,64) 
# you can also change the number of batches (keep the shape of X_train in mind)
X_train_splits = np.array_split(X_train_sc, 96)
y_train_splits = np.array_split(y_train, 96)

# training the model on those minibatches
for X_train_split, y_train_split in zip(X_train_splits, y_train_splits):
    clf.partial_fit(X_train_split, y_train_split, classes= np.unique([1,0]))

# Plotting the loss function
sys.stdout = old_stdout
loss_history = mystdout.getvalue()
loss_list = []
for line in loss_history.split('\n'):
    if(len(line.split("loss: ")) == 1):
        continue
    loss_list.append(float(line.split("loss: ")[-1].split(",")[0]))
plt.figure()
plt.plot(np.arange(len(loss_list)), loss_list)
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

In [ ]:
accuracy = clf.score(X_test_sc, y_test)
print(f"Accuracy: {round(accuracy, 2)}")

Im Vergleich zum SGD können Sie hier sehen, dass der Mini Batch GD scheinbar mehr Iterationen benötigt, um zu einem ähnlichen Ergebnis zu kommen. Dies liegt jedoch am Fehlen eines frühen Stoppens (bei dem das Modell aufhört zu fitten, wenn die Verlustfunktion nicht besser wird) mit ```partial_fit()```.